In [1]:
%pip install nltk xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
# Remove Stopwords
from nltk.corpus import stopwords 


In [3]:
df_train = pd.read_csv("mail_data.csv",encoding="latin-1")

In [4]:
df_train.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [5]:
print(f"Null values in train dataset is:{df_train.isnull().sum().sum()}")

Null values in train dataset is:0


In [6]:
print(f'Dublicate Values in Train dataset is :{df_train.duplicated().sum()}')

Dublicate Values in Train dataset is :415


In [7]:
df_train.drop_duplicates(inplace=True)

In [8]:
#Null Values column
df_train.isnull().sum()

Category    0
Message     0
dtype: int64

In [9]:
print(df_train['Message'][0])
print(df_train['Message'][1])
print(df_train['Message'][2])

Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
Ok lar... Joking wif u oni...
Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's


# Text Preprocessing


In This we do some basic Text Preprocessing Like Cleaning Text, Tokenization etc

In [10]:
# 1.remove lowercase

df_train["Message"]=df_train['Message'].str.lower()

df_train.head()

,Category,Message
0,ham,"go until jurong point, crazy.. available only ..."
1,ham,ok lar... joking wif u oni...
2,spam,free entry in 2 a wkly comp to win fa cup fina...
3,ham,u dun say so early hor... u c already then say...
4,ham,"nah i don't think he goes to usf, he lives aro..."


In [11]:
#2. Remove # tag from the train datasets
df_train['Message']=df_train ['Message'].str.replace('#','')

In [12]:
#3. Remove @ From Train and Test Text
df_train['Message']=df_train['Message'].str.replace('@','')

In [13]:
#4. Remove URL from test and train text
df_train['Message']=df_train['Message'].str.replace(r'^https?:\/\/.*[\r\n]*','')

In [14]:
#5. Removing Punctuations
import string 
df_train['Message'] = df_train['Message'].str.translate(str.maketrans('', '', string.punctuation))

#head
df_train.head()

,Category,Message
0,ham,go until jurong point crazy available only in ...
1,ham,ok lar joking wif u oni
2,spam,free entry in 2 a wkly comp to win fa cup fina...
3,ham,u dun say so early hor u c already then say
4,ham,nah i dont think he goes to usf he lives aroun...


In [15]:
import nltk
# Download the missing stopwords resource
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to C:\Users\ABDUL
[nltk_data]     MAJEED\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [16]:
stop_words = stopwords.words('english')

#Apply stopwords
df_train['Message']=df_train['Message'].apply(lambda x : ' '.join([word for word in x.split()if word not in (stop_words)]))

df_train.head()

,Category,Message
0,ham,go jurong point crazy available bugis n great ...
1,ham,ok lar joking wif u oni
2,spam,free entry 2 wkly comp win fa cup final tkts 2...
3,ham,u dun say early hor u c already say
4,ham,nah dont think goes usf lives around though


In [17]:
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          Trainer, TrainingArguments)

c:\Users\ABDUL MAJEED\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [18]:
# 1. Force reload the original data to get the clean text back
df_train = df_train.rename(
    columns={"v1": "Category", "v2": "Message"}
) 

In [19]:
# 2. Tokenization and Data Split
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

id_map = {"ham": 0, "spam": 1}
df_train["label"] = df_train["Category"].map(id_map)

raw_dataset = Dataset.from_pandas(df_train[["Message", "label"]])


def tokenize_function(example):
    return tokenizer(example["Message"], truncation=True, max_length=128)


tokenized_dataset = raw_dataset.map(
    tokenize_function, batched=True, remove_columns=["Message"]
)
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")

dataset_splits = tokenized_dataset.train_test_split(test_size=0.18, seed=42)
train_dataset = dataset_splits["train"]
eval_dataset = dataset_splits["test"]

Map: 100%|██████████| 5157/5157 [00:00<00:00, 8510.20 examples/s]


In [20]:
# 3. Initialize fresh model
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2,
    id2label={0: "ham", 1: "spam"},
    label2id={"ham": 0, "spam": 1},
)


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}


training_args = TrainingArguments(
    output_dir="./results_final",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    load_best_model_at_end=True,
    logging_steps=50,
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1088.91it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [21]:
# 4. Train
trainer.train()

c:\Users\ABDUL MAJEED\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.065812,0.063130,0.981701,0.928270,0.982143,0.880000
2,0.032893,0.055263,0.986006,0.946502,0.974576,0.920000


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.80s/it]
c:\Users\ABDUL MAJEED\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.01s/it]


TrainOutput(global_step=530, training_loss=0.0747289351697238, metrics={'train_runtime': 1387.1096, 'train_samples_per_second': 6.096, 'train_steps_per_second': 0.382, 'total_flos': 89284061595696.0, 'train_loss': 0.0747289351697238, 'epoch': 2.0})

In [27]:

# 5. Final Inference Test
test_messages = [
    "Congratulations! You have won a FREE iPhone 16 and a cash prize of $5,000. Claim your reward now by clicking this link: www.claim-now-prize.com. Hurry! Offer expires today.",
    "Hey, are we still meeting up for lunch today at 1 PM?",
]

model.eval()
for msg in test_messages:
    inputs = tokenizer(
        msg, truncation=True, max_length=128, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    pred_idx = torch.argmax(probs, dim=-1).item()
    confidence = probs[0][pred_idx].item()

    print(f"Text: {msg}")
    print(
        f"Prediction: {model.config.id2label[pred_idx]} ({confidence*100:.2f}% confidence)\n"
    )

Text: Congratulations! You have won a FREE iPhone 16 and a cash prize of $5,000. Claim your reward now by clicking this link: www.claim-now-prize.com. Hurry! Offer expires today.
Prediction: spam (96.51% confidence)

Text: Hey, are we still meeting up for lunch today at 1 PM?
Prediction: ham (99.42% confidence)



In [ ]:
import os

save_directory = "./results/results_final"
os.makedirs(save_directory, exist_ok=True)

trainer.save_model(save_directory)
tokenizer.save_pretrained(save_directory)

print("✅ Model and Tokenizer sahi jagah save ho chuke hain!")


Writing model shards:   0%|          | 0/1 [00:01<?, ?it/s]


SafetensorError: Error while serializing: I/O error: The requested operation cannot be performed on a file with a user-mapped section open. (os error 1224)